In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

In [2]:
dico_listings = pd.read_csv("../data/raw/Listings_data_dictionary.csv")
dico_listings

,Field,Description
0,listing_id,Listing ID
1,name,Listing Name
2,host_id,Host ID
3,host_since,Date the Host joined Airbnb
4,host_location,Location where the Host is based
5,host_response_time,Estimate of how long the Host takes to respond
6,host_response_rate,Percentage of times the Host responds
7,host_acceptance_rate,Percentage of times the Host accepts a booking request
8,host_is_superhost,Binary field to determine if the Host is a Superhost
9,host_total_listings_count,Total listings the Host has in Airbnb


In [3]:
dico_reviews = pd.read_csv("../data/raw/Reviews_data_dictionary.csv")
dico_reviews

,Field,Description
0,listing_id,Listing ID
1,review_id,Review ID
2,date,Review date
3,reviewer_id,Reviewer ID


In [4]:
listings = pd.read_csv("../data/raw/Listings.csv", encoding="utf-8", encoding_errors="replace", low_memory=False)
listings.shape

(279712, 33)

In [5]:
listings[listings["city"] == "Paris"][["name", "neighbourhood", "room_type", "price", "accommodates", "review_scores_rating"]].head()

,name,neighbourhood,room_type,price,accommodates,review_scores_rating
0,"Beautiful Flat in le Village Montmartre, Paris",Buttes-Montmartre,Entire place,53,2,100.0
1,39 mÂ² Paris (Sacre CÅ“ur),Buttes-Montmartre,Entire place,120,2,100.0
2,"Lovely apartment with Terrace, 60m2",Elysee,Entire place,89,2,100.0
3,Cosy studio (close to Eiffel tower),Vaugirard,Entire place,58,2,100.0
4,Close to Eiffel Tower - Beautiful flat : 2 rooms,Passy,Entire place,60,2,100.0


In [6]:
listings["city"].value_counts()

city
Paris             64690
New York          37012
Sydney            33630
Rome              27647
Rio de Janeiro    26615
Istanbul          24519
Mexico City       20065
Bangkok           19361
Cape Town         19086
Hong Kong          7087
Name: count, dtype: int64

Le fichier contient 279 712 annonces réparties dans 10 villes. Paris est la ville qui en a le plus (64 690). Je garde uniquement Paris car les prix sont dans la monnaie locale de chaque ville.

In [7]:
#on ne garde que les annonces de Paris
listings_paris = listings[listings["city"] == "Paris"].copy()
listings_paris.shape

(64690, 33)

In [8]:
#les types
listings_paris.dtypes

listing_id                       int64
name                               str
host_id                          int64
host_since                         str
host_location                      str
host_response_time                 str
host_response_rate             float64
host_acceptance_rate           float64
host_is_superhost                  str
host_total_listings_count      float64
host_has_profile_pic               str
host_identity_verified             str
neighbourhood                      str
district                           str
city                               str
latitude                       float64
longitude                      float64
property_type                      str
room_type                          str
accommodates                     int64
bedrooms                       float64
amenities                          str
price                            int64
minimum_nights                   int64
maximum_nights                   int64
review_scores_rating     

In [9]:
#valeurs manquantes
listings_paris.isna().sum()

listing_id                         0
name                              63
host_id                            0
host_since                        33
host_location                    168
host_response_time             41344
host_response_rate             41344
host_acceptance_rate           32771
host_is_superhost                 33
host_total_listings_count         33
host_has_profile_pic              33
host_identity_verified            33
neighbourhood                      0
district                       64690
city                               0
latitude                           0
longitude                          0
property_type                      0
room_type                          0
accommodates                       0
bedrooms                       13404
amenities                          0
price                              0
minimum_nights                     0
maximum_nights                     0
review_scores_rating           16654
review_scores_accuracy         16701
r

In [10]:
#doublons
listings_paris.duplicated().sum()

np.int64(0)

In [11]:
#je verifie quand meme si il y'en a de meme type
listings_paris["listing_id"].duplicated().sum()

np.int64(0)

aucuns doublons dans listings_paris

In [12]:
#resume 
listings_paris["price"].describe()

count    64690.000000
mean       113.096445
std        214.433668
min          0.000000
25%         59.000000
50%         80.000000
75%        120.000000
max      12000.000000
Name: price, dtype: float64

In [13]:
#les annonces a 0 
(listings_paris["price"] == 0).sum()

np.int64(62)

In [14]:
#annonces les plus cheres
listings_paris.sort_values("price", ascending=False)[["price", "room_type", "accommodates", "bedrooms", "neighbourhood"]].head(10)

,price,room_type,accommodates,bedrooms,neighbourhood
36228,12000,Entire place,2,1.0,Gobelins
60735,11599,Entire place,10,4.0,Gobelins
227298,10250,Entire place,10,5.0,Passy
28872,9631,Entire place,4,1.0,Temple
64147,9280,Entire place,6,2.0,Reuilly
60860,9280,Entire place,10,4.0,Gobelins
60845,9280,Entire place,12,4.0,Opera
45174,9280,Entire place,2,NaN,Buttes-Chaumont
45175,9280,Entire place,2,NaN,Buttes-Chaumont
225899,9280,Entire place,4,1.0,Enclos-St-Laurent


In [15]:
print("Prix à 0 € :", (listings_paris["price"] == 0).sum())
print("Prix > 500 € :", (listings_paris["price"] > 500).sum())
print("Prix > 1 000 € :", (listings_paris["price"] > 1000).sum())
print("99 % des annonces coûtent moins de :", listings_paris["price"].quantile(0.99), "€")

Prix à 0 € : 62
Prix > 500 € : 823
Prix > 1 000 € : 192
99 % des annonces coûtent moins de : 600.0 €


In [16]:
#connaitre les valeurs avant connversion et verifier que les deux groupes surperhost et non superhost sont assez grand pour faire des comparaisons
listings_paris["host_is_superhost"].value_counts()

host_is_superhost
f    56545
t     8112
Name: count, dtype: int64

In [17]:
#connaitre les types de logement et reperer les valeurs aberrantes 
listings_paris["room_type"].value_counts()

room_type
Entire place    55378
Private room     7492
Hotel room       1396
Shared room       424
Name: count, dtype: int64

In [18]:
reviews = pd.read_csv("../data/raw/Reviews.csv", encoding="utf-8", encoding_errors="replace", low_memory=False)
reviews.shape

(5373143, 4)

In [19]:
reviews.head()

,listing_id,review_id,date,reviewer_id
0,11798,330265172,2018-09-30,11863072
1,15383,330103585,2018-09-30,39147453
2,16455,329985788,2018-09-30,1125378
3,17919,330016899,2018-09-30,172717984
4,26827,329995638,2018-09-30,17542859


In [20]:
#on ne garde que les reviews des annonces de Paris
reviews_paris = reviews[reviews["listing_id"].isin(listings_paris["listing_id"])].copy()
reviews_paris.shape

(1213727, 4)

In [21]:
#les types des colonnes
reviews_paris.dtypes

listing_id     int64
review_id      int64
date             str
reviewer_id    int64
dtype: object

In [22]:
#valeurs manquantes par colonnes
reviews_paris.isna().sum()

listing_id     0
review_id      0
date           0
reviewer_id    0
dtype: int64

In [23]:
#doublons
reviews_paris["review_id"].duplicated().sum()

np.int64(0)

Constats de l’exploration

Annonces (Listings.csv):
Le fichier contient 279 712 annonces dans 10 villes. Je garde uniquement Paris (64 690 annonces), car les prix sont dans la monnaie de chaque ville.
La colonne district est vide à 100 % pour Paris.
Beaucoup de valeurs manquent pour le temps et le taux de réponse des hôtes (51 à 64 %), pour les notes (26 %) et pour le nombre de chambres (21 %). 33 annonces n’ont aucune information sur l’hôte.
Certains titres d’annonces ont des caractères mal encodés dès la source (par exemple « mÂ² » au lieu de « m² »).
Aucun doublon.
Les prix vont de 0 € à 12 000 € la nuit, alors que 99 % des annonces coûtent moins de 600 €. 62 annonces sont à 0 € et 192 dépassent 1 000 €.
Plusieurs colonnes ont un mauvais type : host_since devrait être une date, et les colonnes en t/f (comme host_is_superhost) devraient être des vrai/faux.

Avis (Reviews.csv):
Le fichier contient 5 373 143 avis. En gardant ceux des annonces de Paris, il en reste 1 213 727.
Aucune valeur manquante et aucun doublon.
La colonne date est du texte  il faudra la convertir en date pour l’analyse par mois.